# S5 J2 — API FastAPI

Notebook étudiant généré à partir du Markdown source.

## Objectifs

# Objectifs pédagogiques

À la fin de cette journée, l'apprenant doit être capable de :

1. expliquer le rôle de FastAPI dans une plateforme IA de production ;
2. concevoir une frontière claire entre transport HTTP, service applicatif et runtime agentique ;
3. créer des contrats Pydantic stricts pour les entrées et sorties ;
4. structurer une application avec `FastAPI`, `APIRouter`, dépendances et middleware ;
5. exposer des endpoints `/healthz`, `/readyz`, `/v1/chat` et `/v1/sessions/{session_id}` ;
6. appliquer une authentification simple par `X-API-Key` ;
7. propager un `X-Request-ID` pour faciliter le debug production ;
8. stabiliser les erreurs HTTP dans une enveloppe JSON exploitable ;
9. tester une API FastAPI avec `TestClient` sans lancer de serveur réseau ;
10. préserver une API publique Python cumulative dans `ai_platform/__init__.py`.

## Compétences AI Engineering

- API design pour workloads LLM.
- Validation avant appel modèle.
- Isolation de session.
- Rate limiting applicatif.
- Redaction PII.
- Observabilité minimale par request ID.
- Tests d'intégration HTTP.


## Chapitre

# Chapitre — Construire une API FastAPI pour une plateforme IA

## 1. Pourquoi une API dédiée pour l'IA ?

Un prototype d'agent peut être lancé depuis un notebook ou un script.  
Un système de production doit être exposé via une interface stable, testable et observable.

L'API n'est pas seulement un proxy vers un modèle. Elle est responsable de :

- valider les entrées ;
- authentifier le client ;
- appliquer des limites ;
- propager un identifiant de requête ;
- isoler les sessions ;
- transformer les erreurs internes en erreurs HTTP stables ;
- appeler le service applicatif ;
- retourner une réponse structurée.

Dans une architecture IA backend, la couche API doit rester mince.  
Elle ne doit pas contenir la logique métier profonde ni la boucle agentique complète.

```mermaid
flowchart LR
    Client[Client web ou backend] --> API[FastAPI HTTP API]
    API --> Validation[Validation Pydantic]
    Validation --> Service[SupportAIService]
    Service --> Agent[Runtime agentique]
    Agent --> Tools[Tools / MCP / DB]
    API --> Obs[Request ID / Logs / Metrics]
```

## 2. Séparer transport et métier

Une erreur fréquente consiste à écrire toute la logique dans la fonction de route :

```python
@app.post("/chat")
def chat(payload):
    # validation manuelle
    # prompt building
    # appel modèle
    # appel base
    # logs
    # réponse
```

Cette approche rend l'API difficile à tester et à faire évoluer.

Le design recommandé pour le bootcamp est :

```text
HTTP route
  -> validation Pydantic
  -> dépendances FastAPI
  -> service applicatif
  -> runtime agentique
  -> réponse Pydantic
```

Le jour 2 introduit donc `ai_platform/api.py`, qui expose :

- `APISettings` ;
- `ChatRequest` ;
- `ChatResponse` ;
- `ErrorResponse` ;
- `InMemoryRateLimiter` ;
- `DeterministicSupportService` ;
- `create_app()`.

## 3. Contrats Pydantic

Les modèles Pydantic sont le premier garde-fou de production.  
Ils permettent de refuser les entrées mal formées avant toute exécution coûteuse.

Exemple de contrat d'entrée :

```python
class ChatRequest(BaseModel):
    session_id: str
    user_id: str
    message: str
    metadata: dict = {}
```

Dans le lab, le modèle est plus strict :

- taille minimale et maximale ;
- identifiants sans caractères dangereux ;
- champs supplémentaires interdits ;
- message non vide.

Cette rigueur est importante pour les systèmes IA : un LLM peut tolérer beaucoup d'ambiguïté, mais une API de production doit être explicite.

## 4. Endpoints de production

Le lab expose quatre endpoints :

| Endpoint | Rôle | Authentification |
|---|---|---|
| `GET /healthz` | Vérifie que le processus répond | Non |
| `GET /readyz` | Vérifie que le service applicatif est prêt | Non |
| `POST /v1/chat` | Exécute un tour assistant | Oui |
| `GET /v1/sessions/{session_id}` | Relit la session stockée | Oui |

La séparation `healthz` / `readyz` est volontaire :

- `healthz` répond si l'application fonctionne ;
- `readyz` répond si l'application peut traiter du trafic.

## 5. Authentification via dépendance FastAPI

FastAPI permet d'ajouter des dépendances à un routeur complet.

Dans le lab :

```python
router = APIRouter(
    prefix="/v1",
    tags=["ai"],
    dependencies=[Depends(require_api_key)],
)
```

Cela évite de répéter la vérification de clé API sur chaque endpoint protégé.

Cette clé API est volontairement simple pour le lab.  
En production, on attendrait plutôt une intégration OAuth2, JWT, mTLS, gateway API ou service mesh selon le contexte.

## 6. Request ID

Un système IA est difficile à diagnostiquer sans corrélation.

Le middleware du lab :

- lit `X-Request-ID` ou `X-Client-Request-ID` ;
- génère un UUID si absent ;
- stocke l'identifiant dans `request.state.request_id` ;
- renvoie le même identifiant dans la réponse.

Cette pratique permet de relier :

- requête HTTP ;
- appel modèle ;
- appels outils ;
- logs ;
- traces ;
- ticket support.

## 7. Erreurs stables

Un client ne doit pas parser des erreurs hétérogènes.  
Le lab transforme les `HTTPException` en enveloppe stable :

```json
{
  "error": "401",
  "message": "invalid or missing API key",
  "request_id": "req-123"
}
```

L'objectif n'est pas d'effacer le code HTTP.  
L'objectif est d'ajouter une structure exploitable côté client et côté observabilité.

## 8. Rate limiting pédagogique

Le lab inclut un rate limiter mémoire limité par `session_id`.

Ce n'est pas un composant de production distribué.  
Il sert à montrer où placer ce contrôle et comment le tester.

En production, le rate limiting serait généralement porté par :

- API gateway ;
- Redis ;
- reverse proxy ;
- service mesh ;
- quota par organisation ;
- budget par utilisateur.

## 9. Redaction PII

Le service déterministe masque les emails et numéros de carte détectés avant stockage dans la transcription.

Cette logique est simplifiée, mais elle installe un réflexe clé :  
ne pas stocker aveuglément ce que l'utilisateur envoie.

## 10. Tests

Le lab teste notamment :

- endpoints publics ;
- authentification ;
- validation ;
- limitation de taille ;
- rate limiting ;
- redaction PII ;
- action sensible ;
- OpenAPI ;
- exports cumulatifs du package.

La commande principale est :

```bash
python book/week05/day02/labs/test_api_fastapi.py
```

## À retenir

Une API IA de production ne doit pas être un simple wrapper autour d'un appel LLM.

Elle doit être :

- contractuelle ;
- testée ;
- observable ;
- sécurisée ;
- limitée ;
- claire dans ses erreurs ;
- indépendante de l'implémentation interne de l'agent.


*Suite complète dans `book/week05/day02/chapter.md`.*

## Exercices

# Exercices

## Exercice 1 — Identifier les responsabilités de la couche API

Classe les responsabilités suivantes en deux catégories : `API HTTP` ou `service applicatif`.

1. Lire l'en-tête `X-API-Key`.
2. Classer l'intention utilisateur.
3. Retourner un code `401`.
4. Stocker un tour dans une session.
5. Valider la taille du message.
6. Générer la réponse métier.
7. Renvoyer un `X-Request-ID`.
8. Masquer les emails avant stockage.

## Exercice 2 — Étendre le contrat `ChatRequest`

Ajoute mentalement un champ `channel` autorisé parmi :

- `web`
- `mobile`
- `slack`
- `api`

Explique :

1. où le champ doit être validé ;
2. pourquoi il ne doit pas être laissé comme chaîne libre ;
3. comment il peut aider l'observabilité.

## Exercice 3 — Concevoir une erreur stable

Propose une enveloppe JSON pour une erreur `429 Too Many Requests`.

Elle doit contenir au minimum :

- code d'erreur ;
- message lisible ;
- request ID ;
- limite ;
- nombre restant.

## Exercice 4 — Tester sans serveur réseau

Explique pourquoi `TestClient` est adapté pour tester une API FastAPI pendant le développement.

Donne deux cas qui doivent être testés en HTTP et non uniquement au niveau fonction Python.

## Exercice 5 — API publique cumulative

Explique pourquoi le fichier `ai_platform/__init__.py` du jour 2 doit conserver les exports du jour 1.

Que risque-t-on si chaque jour réécrit ce fichier de façon isolée ?


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from ai_platform.api import APISettings, DeterministicSupportService, InMemoryRateLimiter, create_app
print(APISettings(require_api_key=False).model_dump())

In [ ]:
from fastapi.testclient import TestClient
client = TestClient(create_app(APISettings(require_api_key=False)))
response = client.post('/v1/chat', json={'session_id':'nb-session','user_id':'nb-user','message':'billing issue with invoice'})
response.status_code, response.json()

## Challenge

# Challenge — API de support IA production-ready

## Objectif

Étendre le lab pour rendre l'API plus proche d'un environnement production.

## Contexte

Tu disposes d'une API FastAPI exposant :

- `GET /healthz`
- `GET /readyz`
- `POST /v1/chat`
- `GET /v1/sessions/{session_id}`

## Travail demandé

Ajoute une route :

```text
POST /v1/admin/sessions/{session_id}/forget
```

Cette route doit :

1. être protégée par la clé API ;
2. demander un en-tête `X-Human-Approval: approved` ;
3. supprimer la session si elle existe ;
4. retourner une réponse structurée ;
5. retourner `404` si la session n'existe pas ;
6. retourner `403` si l'approbation humaine est absente ;
7. écrire une trace ou un événement d'audit simplifié.

## Contraintes

- Ne pas changer l'arborescence du projet.
- Ne pas supprimer les exports existants.
- Ne pas appeler de service externe.
- Ajouter des tests HTTP.
- Garder une réponse JSON stable.

## Critères d'acceptation

Le challenge est réussi si :

- les tests existants passent toujours ;
- la nouvelle route est visible dans OpenAPI ;
- une session oubliée n'est plus lisible ;
- une suppression sans approbation est refusée ;
- l'API publique reste cumulative.


# Corrigés

## Exercices

# Corrigé — Exercices

## Exercice 1

| Responsabilité | Catégorie | Explication |
|---|---|---|
| Lire `X-API-Key` | API HTTP | C'est une précondition d'accès au routeur protégé. |
| Classer l'intention | Service applicatif | C'est une décision métier ou agentique. |
| Retourner `401` | API HTTP | La route transforme une erreur d'accès en réponse HTTP. |
| Stocker un tour | Service applicatif | Le stockage de session appartient à l'état applicatif. |
| Valider la taille | API HTTP | La limite protège l'entrée avant traitement. |
| Générer la réponse | Service applicatif | La réponse dépend du domaine, pas du transport. |
| Renvoyer `X-Request-ID` | API HTTP | C'est une responsabilité de middleware. |
| Masquer les emails | Service applicatif | Dans le lab, la redaction est faite avant stockage métier. |

## Exercice 2

Le champ `channel` doit être validé dans le modèle Pydantic, par exemple avec un `Literal`.

Il ne doit pas rester une chaîne libre parce que :

- les dashboards deviendraient incohérents ;
- les clients pourraient envoyer des valeurs non prévues ;
- les règles métier par canal deviendraient fragiles.

Il aide l'observabilité en permettant de comparer les erreurs, latences et volumes par canal.

## Exercice 3

Exemple :

```json
{
  "error": "429",
  "message": "rate limit exceeded for session session-123",
  "request_id": "req-001",
  "limit": 5,
  "remaining": 0
}
```

## Exercice 4

`TestClient` est adapté parce qu'il exécute l'application FastAPI directement, sans serveur réseau externe.

Deux cas à tester en HTTP :

1. l'authentification par en-tête ;
2. la validation Pydantic et les codes d'erreur.

Ces comportements dépendent de FastAPI et ne sont pas visibles dans un simple test de fonction métier.

## Exercice 5

`ai_platform/__init__.py` est la façade publique du package.

Si le jour 2 remplace les exports du jour 1, le code suivant peut casser :

```python
from ai_platform import ArchitectureBlueprint
```

La construction du framework serait alors non cumulative, ce qui contredit la progression pédagogique de la semaine.


## Entretien

# Corrigé — Questions d'entretien

## Réponse 1

Mettre la logique agentique dans les routes mélange transport, métier et orchestration.  
Cela rend le code difficile à tester, à monitorer, à réutiliser et à remplacer.

Une route doit valider, contrôler et déléguer.

## Réponse 2

`healthz` vérifie que le processus répond.  
`readyz` vérifie que l'application peut réellement traiter du trafic.

Une API peut être vivante mais non prête si une dépendance critique est indisponible.

## Réponse 3

Un `request_id` permet de corréler la requête HTTP, les logs, les appels modèle, les appels outils, les erreurs et les traces.  
Sans cet identifiant, le debug production devient lent et imprécis.

## Réponse 4

En entreprise, on peut utiliser OAuth2, JWT, mTLS, API gateway, IAM cloud ou service mesh.  
Le choix dépend du périmètre : utilisateurs finaux, services internes, partenaires ou backoffice.

## Réponse 5

Valider avant d'appeler un modèle réduit les coûts, évite les entrées dangereuses, améliore les erreurs client et protège les dépendances internes.

## Réponse 6

Un rate limiter en mémoire ne fonctionne pas correctement avec plusieurs processus, plusieurs pods ou plusieurs régions.  
Il ne survit pas au redémarrage et ne permet pas une gouvernance centralisée.

## Réponse 7

Tester OpenAPI permet de vérifier que le contrat public de l'API expose bien les routes prévues.  
C'est utile pour les clients générés, la documentation et les gateways.

## Réponse 8

Une erreur utilisateur vient d'une requête invalide.  
Une erreur de modèle vient du fournisseur ou de la sortie du modèle.  
Une erreur d'outil vient d'une dépendance appelée par l'agent.

Les trois doivent être distinguées pour diagnostiquer correctement le système.


## Challenge

# Corrigé — Challenge

## Approche recommandée

Ajouter une route admin dans le routeur `/v1`.

Pseudo-code :

```python
@router.post("/admin/sessions/{session_id}/forget")
def forget_session(
    session_id: str,
    x_human_approval: str | None = Header(default=None, alias="X-Human-Approval"),
):
    if x_human_approval != "approved":
        raise HTTPException(status_code=403, detail="human approval required")

    if session_id not in service.sessions:
        raise HTTPException(status_code=404, detail="session not found")

    del service.sessions[session_id]
    audit_log.append({
        "event": "session_forgotten",
        "session_id": session_id,
    })
    return {
        "status": "forgotten",
        "session_id": session_id,
    }
```

## Tests attendus

1. Créer une session avec `/v1/chat`.
2. Appeler `forget` sans `X-Human-Approval` : attendre `403`.
3. Appeler `forget` avec `X-Human-Approval: approved` : attendre `200`.
4. Relire la session : attendre `404`.
5. Vérifier que `/openapi.json` expose la route.

## Point d'architecture

L'oubli utilisateur est une action sensible.  
Elle doit être protégée par :

- authentification ;
- approbation humaine ou workflow équivalent ;
- audit log ;
- résultat structuré ;
- tests de non-régression.


## Review formateur

# Review formateur

## Ce que l'apprenant doit avoir compris

- FastAPI est une frontière de production, pas une simple démo HTTP.
- Les schémas Pydantic protègent le système avant l'appel modèle.
- Les dépendances FastAPI permettent de centraliser auth et contrôles.
- Le request ID est indispensable au diagnostic production.
- Les erreurs doivent être stables pour les clients.
- Un rate limiter mémoire est pédagogique, pas distribué.
- `ai_platform/__init__.py` doit rester cumulatif.

## Points d'attention

- Ne pas mélanger logique agentique et fonction de route.
- Ne pas stocker de PII brute.
- Ne pas retourner les exceptions Python telles quelles.
- Ne pas introduire de dépendance externe non maîtrisée dans les tests.
- Ne pas casser les exports du jour 1.

## Questions de validation orale

1. Où placerais-tu un appel à Redis pour le rate limiting ?
2. Où placerais-tu l'appel au modèle OpenAI ?
3. Pourquoi `ChatResponse` doit-il être stable ?
4. Quelle donnée mettrais-tu dans les logs et quelle donnée éviterais-tu ?
5. Comment versionnerais-tu `/v1/chat` si le contrat change ?

## Propositions d'amélioration

Ces propositions ne modifient pas les spécifications figées :

- ajouter une gateway locale dans une future journée production ;
- ajouter un module `ai_platform/security.py` au jour sécurité ;
- ajouter une stratégie de versioning d'API plus avancée ;
- brancher l'observabilité de la semaine 4 sur les middlewares API.
